# Portfolio ML - Summary Dashboard

Notebook de seguimiento interactivo del pipeline completo:
1. Descarga de datos (`raw`)
2. Limpieza y panel de features (`processed`)
3. Entrenamiento walk-forward (`outputs/models`)
4. Backtest y riesgo (`outputs/backtests`)

El objetivo es inspeccionar resultados sin volver a ejecutar todo el automatismo.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('ggplot')
pd.options.display.max_columns = 200
pd.options.display.width = 140


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / 'configs').exists() and (p / 'src').exists():
            return p
    raise RuntimeError('No se pudo detectar la raiz del proyecto.')

ROOT = find_project_root()
PATHS = {
    'raw': ROOT / 'data/raw/prices.parquet',
    'clean': ROOT / 'data/processed/prices_clean.parquet',
    'panel': ROOT / 'data/processed/panel.parquet',
    'train_summary': ROOT / 'outputs/models/train_summary.json',
    'predictions': ROOT / 'outputs/models/predictions_oos.parquet',
    'training_log': ROOT / 'outputs/models/training_log.parquet',
    'feature_importance': ROOT / 'outputs/models/feature_importance.parquet',
    'backtest_summary': ROOT / 'outputs/backtests/backtest_summary.json',
    'daily_returns': ROOT / 'outputs/backtests/daily_returns.parquet',
    'rebalance_log': ROOT / 'outputs/backtests/rebalance_log.parquet',
    'weights_history': ROOT / 'outputs/backtests/weights_history.parquet',
}

print(f'Project root: {ROOT}')

In [ ]:
status = []
for name, path in PATHS.items():
    status.append({
        'artifact': name,
        'exists': path.exists(),
        'path': str(path),
        'size_kb': round(path.stat().st_size / 1024, 2) if path.exists() else None,
    })

status_df = pd.DataFrame(status).sort_values('artifact').reset_index(drop=True)
display(status_df)

## 1) Raw Data (descarga)

In [ ]:
raw = pd.read_parquet(PATHS['raw'])
raw['date'] = pd.to_datetime(raw['date'])

print('Shape:', raw.shape)
print('Date range:', raw['date'].min().date(), '->', raw['date'].max().date())
print('Tickers:', raw['ticker'].nunique())
print('Missing % by column:')
display((raw.isna().mean() * 100).round(2).rename('pct_missing').to_frame())

sample_tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL']
plot_df = raw[raw['ticker'].isin(sample_tickers)].pivot(index='date', columns='ticker', values='adj_close').sort_index()
norm = plot_df / plot_df.ffill().bfill().iloc[0]

ax = norm.plot(figsize=(12, 5), linewidth=1.6, title='Raw Adj Close (base=1)')
ax.set_ylabel('Normalized price')
ax.set_xlabel('Date')
plt.show()

## 2) Clean Data + Feature Panel

In [ ]:
clean = pd.read_parquet(PATHS['clean'])
panel = pd.read_parquet(PATHS['panel'])
clean['date'] = pd.to_datetime(clean['date'])
panel['date'] = pd.to_datetime(panel['date'])

print('Clean shape:', clean.shape)
print('Panel shape:', panel.shape)
print('Panel date range:', panel['date'].min().date(), '->', panel['date'].max().date())
print('Panel tickers:', panel['ticker'].nunique())

display(panel.describe(include='all').T[['count', 'mean', 'std', 'min', 'max']].head(12))

features = ['ret_1d', 'ret_5d', 'ret_20d', 'vol_20d', 'momentum_20_60']
corr = panel[features + ['fwd_return_5d']].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title('Correlation Matrix (features + target)')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 3) Walk-Forward XGBoost

In [ ]:
with PATHS['train_summary'].open('r', encoding='utf-8') as fh:
    train_summary = json.load(fh)

training_log = pd.read_parquet(PATHS['training_log'])
feature_importance = pd.read_parquet(PATHS['feature_importance'])
pred = pd.read_parquet(PATHS['predictions'])

training_log['rebalance_date'] = pd.to_datetime(training_log['rebalance_date'])

display(pd.DataFrame([train_summary]).T.rename(columns={0: 'value'}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(training_log['rebalance_date'], training_log['validation_ic_spearman'], linewidth=1.2)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_title('Validation IC (Spearman) by rebalance')
axes[0].set_xlabel('Rebalance date')
axes[0].set_ylabel('IC')

fi = feature_importance.groupby('feature', as_index=False)['importance'].mean().sort_values('importance')
axes[1].barh(fi['feature'], fi['importance'])
axes[1].set_title('Mean Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

oos_ic = pred['prediction'].corr(pred['fwd_return_5d'], method='spearman')
print('OOS IC recomputed:', round(float(oos_ic), 6))

## 4) Backtest, Riesgo y Cartera

In [ ]:
with PATHS['backtest_summary'].open('r', encoding='utf-8') as fh:
    backtest_summary = json.load(fh)

daily = pd.read_parquet(PATHS['daily_returns'])
rebalance_log = pd.read_parquet(PATHS['rebalance_log'])
weights = pd.read_parquet(PATHS['weights_history'])

daily['date'] = pd.to_datetime(daily['date'])
rebalance_log['rebalance_date'] = pd.to_datetime(rebalance_log['rebalance_date'])
weights['rebalance_date'] = pd.to_datetime(weights['rebalance_date'])

display(pd.DataFrame([backtest_summary]).T.rename(columns={0: 'value'}))

equity = (1 + daily['portfolio_return_net']).cumprod()
rolling_max = equity.cummax()
drawdown = equity / rolling_max - 1

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)
axes[0].plot(daily['date'], equity, linewidth=1.4)
axes[0].set_title('Equity Curve (Net)')
axes[0].set_ylabel('Equity')

axes[1].plot(daily['date'], drawdown, color='firebrick', linewidth=1.2)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown')

axes[2].plot(rebalance_log['rebalance_date'], rebalance_log['turnover'], linewidth=1.2)
axes[2].set_title('Turnover by Rebalance')
axes[2].set_ylabel('Turnover')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

latest_reb = weights['rebalance_date'].max()
latest_w = weights[weights['rebalance_date'] == latest_reb].sort_values('weight', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(latest_w['ticker'], latest_w['weight'])
ax.set_title(f'Top 10 Weights @ {latest_reb.date()}')
ax.set_ylabel('Weight')
ax.set_xlabel('Ticker')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

display(latest_w.reset_index(drop=True))

## 5) Checklist Rápido

- Si falta algún artefacto, ejecutar scripts en orden: `00 -> 01 -> 02 -> 03`.
- Si cambias frecuencia de rebalanceo o universo, vuelve a correr `02_train.py` y `03_backtest.py`.
- Este notebook es de inspección y reporting, no de ejecución del pipeline completo.

## 6) Execution / Rebalance

Inspeccion del ultimo plan de ordenes y estado de paper broker.

In [ ]:
exec_dir = ROOT / 'outputs/execution'
summary_files = sorted(exec_dir.glob('*_summary.json'))
orders_files = sorted(exec_dir.glob('*_orders.csv'))

if not summary_files:
    print('No hay summaries de rebalance en outputs/execution.')
else:
    latest_summary_path = summary_files[-1]
    with latest_summary_path.open('r', encoding='utf-8') as fh:
        latest_summary = json.load(fh)
    print('Latest summary:', latest_summary_path.name)
    display(pd.DataFrame([latest_summary]).T.rename(columns={0: 'value'}))

if not orders_files:
    print('No hay order plans en outputs/execution.')
else:
    latest_orders_path = orders_files[-1]
    orders = pd.read_csv(latest_orders_path)
    print('Latest orders:', latest_orders_path.name)
    display(orders.head(20))
    if not orders.empty and 'abs_weight_change' in orders.columns:
        top_orders = orders.sort_values('abs_weight_change', ascending=False).head(10)
    else:
        top_orders = orders.head(10)

    if not top_orders.empty and 'delta_qty' in top_orders.columns:
        colors = top_orders['delta_qty'].apply(lambda x: '#2E8B57' if x > 0 else '#B22222')
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(top_orders['ticker'], top_orders['delta_qty'], color=colors)
        ax.set_title('Top order deltas (shares)')
        ax.set_ylabel('Delta qty')
        ax.set_xlabel('Ticker')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

paper_state = exec_dir / 'paper_state.json'
if paper_state.exists():
    with paper_state.open('r', encoding='utf-8') as fh:
        state = json.load(fh)
    positions = state.get('positions', {})
    pos_df = pd.DataFrame([
        {'ticker': k, **v} for k, v in positions.items()
    ])
    print('Paper state timestamp:', state.get('last_updated'))
    print('Paper cash:', state.get('cash'))
    if not pos_df.empty:
        display(pos_df.sort_values('quantity', ascending=False).head(15))
else:
    print('No existe paper_state.json aun.')
